In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
import pandas as pd
import numpy as np

In [2]:
from src.feature_engineering import FeatureEngineer

fe = FeatureEngineer()


In [3]:
df = pd.read_csv(r"D:\GIT\data-analytics-portfolio\02-telco-customer-churn\data\clean_data\telco_customer_churn_cleaned.csv")

df.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
print(df.info())
print(df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerid        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   seniorcitizen     7043 non-null   int64  
 3   partner           7043 non-null   str    
 4   dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   phoneservice      7043 non-null   str    
 7   multiplelines     7043 non-null   str    
 8   internetservice   7043 non-null   str    
 9   onlinesecurity    7043 non-null   str    
 10  onlinebackup      7043 non-null   str    
 11  deviceprotection  7043 non-null   str    
 12  techsupport       7043 non-null   str    
 13  streamingtv       7043 non-null   str    
 14  streamingmovies   7043 non-null   str    
 15  contract          7043 non-null   str    
 16  paperlessbilling  7043 non-null   str    
 17  paymen

In [5]:
# create customer lifestyle features

df = fe.create_tenure_group(df)

df["tenure_group"].value_counts()


tenure_group
loyal          2239
new            2186
established    1594
developing     1024
Name: count, dtype: int64

In [6]:
# pricing features

df = fe.create_charge_band(df)

df["charge_band"].value_counts()

charge_band
medium     1766
low        1762
premium    1758
high       1757
Name: count, dtype: int64

In [7]:
# Charge per month and additional  service features
df = fe.create_charge_per_month_feature(df)

df[[
    "totalcharges",
    "tenure",
    "charge_per_month_tenure"
]].head()

,totalcharges,tenure,charge_per_month_tenure
0,29.85,1,14.925000
1,1889.50,34,53.985714
2,108.15,2,36.050000
3,1840.75,45,40.016304
4,151.65,2,50.550000


In [8]:
df = fe.create_service_count(df)

df["num_additional_services"].describe()

count    7043.000000
mean        2.037910
std         1.847682
min         0.000000
25%         0.000000
50%         2.000000
75%         3.000000
max         6.000000
Name: num_additional_services, dtype: float64

In [9]:
# verify workflow

df_pipeline = fe.transform(df.copy())

In [10]:
# evaluating relationship between features and target variable

pd.crosstab(
    df["tenure_group"],
    df["churn"],
    normalize="index"
)


churn,No,Yes
tenure_group,,
new,0.525618,0.474382
developing,0.712891,0.287109
established,0.796110,0.203890
loyal,0.904868,0.095132


In [11]:
pd.crosstab(
    df["charge_band"],
    df["churn"],
    normalize="index"
)


churn,No,Yes
charge_band,,
low,0.887628,0.112372
medium,0.754247,0.245753
high,0.624929,0.375071
premium,0.671217,0.328783


In [12]:
pd.crosstab(
    df["num_additional_services"],
    df["churn"],
    normalize="index"
)

churn,No,Yes
num_additional_services,,
0,0.785940,0.214060
1,0.542443,0.457557
2,0.641820,0.358180
3,0.726297,0.273703
4,0.776995,0.223005
5,0.875657,0.124343
6,0.947183,0.052817


## Feature Evaluation Observations

- **Tenure Group:** New customers have the highest churn rate (47.4%), while loyal customers have the lowest (9.5%). Churn decreases consistently as customer tenure increases.

- **Charge Band:** Customers in higher pricing tiers show greater churn rates than those in lower tiers. The highest churn rate is observed among high-charge customers (37.5%).

- **Additional Services:** Customers with more additional services generally exhibit lower churn rates. Churn falls to 5.3% among customers subscribed to six additional services, suggesting that greater service adoption is associated with stronger retention.

In [13]:
ids = df["customerid"]

df_model = fe.encode_features(
    df.drop(columns=["customerid"])
)

df_model.head()

,seniorcitizen,tenure,monthlycharges,totalcharges,charge_per_month_tenure,num_additional_services,gender_Male,partner_Yes,dependents_Yes,phoneservice_Yes,...,paymentmethod_Credit card (automatic),paymentmethod_Electronic check,paymentmethod_Mailed check,churn_Yes,tenure_group_developing,tenure_group_established,tenure_group_loyal,charge_band_medium,charge_band_high,charge_band_premium
0,0,1,29.85,29.85,14.925000,1,False,True,False,False,...,False,True,False,False,False,False,False,False,False,False
1,0,34,56.95,1889.50,53.985714,2,True,False,False,True,...,False,False,True,False,False,True,False,True,False,False
2,0,2,53.85,108.15,36.050000,2,True,False,False,True,...,False,False,True,True,False,False,False,True,False,False
3,0,45,42.30,1840.75,40.016304,3,True,False,False,False,...,False,False,False,False,False,True,False,True,False,False
4,0,2,70.70,151.65,50.550000,0,False,False,False,True,...,False,True,False,True,False,False,False,False,True,False


In [14]:
df_model.select_dtypes(
    include="object"
).columns


Index([], dtype='str')

### Categorical Encoding Check

No object data types remain in the dataset after encoding, indicating that all categorical variables were successfully converted into numerical features suitable for machine learning models.

In [15]:
[col for col in df_model.columns if "churn" in col.lower()]

['churn_Yes']

In [16]:
df_model["churn_Yes"].value_counts()

churn_Yes
False    5174
True     1869
Name: count, dtype: int64

In [17]:
df_model["churn_Yes"].unique()


array([False,  True])

In [18]:
df_model = df_model.rename(
    columns={"churn_Yes": "churn"}
)

df_model["churn"].value_counts()

churn
False    5174
True     1869
Name: count, dtype: int64

In [19]:
# feature validation

df_model.isna().sum().sum()


np.int64(0)

In [20]:
df_model.duplicated().sum()

np.int64(22)

In [21]:
df_model.dtypes

seniorcitizen                              int64
tenure                                     int64
monthlycharges                           float64
totalcharges                             float64
charge_per_month_tenure                  float64
num_additional_services                    int64
gender_Male                                 bool
partner_Yes                                 bool
dependents_Yes                              bool
phoneservice_Yes                            bool
multiplelines_No phone service              bool
multiplelines_Yes                           bool
internetservice_Fiber optic                 bool
internetservice_No                          bool
onlinesecurity_No internet service          bool
onlinesecurity_Yes                          bool
onlinebackup_No internet service            bool
onlinebackup_Yes                            bool
deviceprotection_No internet service        bool
deviceprotection_Yes                        bool
techsupport_No inter

In [22]:
df_model.isna().sum().sort_values(ascending=False)

seniorcitizen                            0
tenure                                   0
monthlycharges                           0
totalcharges                             0
charge_per_month_tenure                  0
num_additional_services                  0
gender_Male                              0
partner_Yes                              0
dependents_Yes                           0
phoneservice_Yes                         0
multiplelines_No phone service           0
multiplelines_Yes                        0
internetservice_Fiber optic              0
internetservice_No                       0
onlinesecurity_No internet service       0
onlinesecurity_Yes                       0
onlinebackup_No internet service         0
onlinebackup_Yes                         0
deviceprotection_No internet service     0
deviceprotection_Yes                     0
techsupport_No internet service          0
techsupport_Yes                          0
streamingtv_No internet service          0
streamingtv

In [24]:
df_model = df_model.drop_duplicates()

In [25]:
df_model[df_model.duplicated()]

,seniorcitizen,tenure,monthlycharges,totalcharges,charge_per_month_tenure,num_additional_services,gender_Male,partner_Yes,dependents_Yes,phoneservice_Yes,...,paymentmethod_Credit card (automatic),paymentmethod_Electronic check,paymentmethod_Mailed check,churn,tenure_group_developing,tenure_group_established,tenure_group_loyal,charge_band_medium,charge_band_high,charge_band_premium


In [26]:
df_model.dtypes.value_counts()

bool       33
int64       3
float64     3
Name: count, dtype: int64

In [27]:
df_model.to_csv(r"D:\GIT\data-analytics-portfolio\02-telco-customer-churn\data\processed\telco_customer_churn_model_data.csv", index=False)